# Flood Detection — Inference & Submission
### Prithvi-EO-2.0-300M-TL-Sen1Floods11 (Full Finetuned Transfer) | AISEHack Theme 1

**Purpose:** Load a trained checkpoint from `stage2-prithvi-v2-300m-tl-flood` and generate `submission.csv`.

**Architecture:** ViT-Large (1024-dim, 24 layers, patch_size=16) + UperNet + SegmentationHead

**Key difference from 600M inference:** This model was pretrained with encoder + decoder + head on Sen1Floods11, so all components carry learned flood features.

**Instructions:**
1. Upload your trained checkpoint (`.ckpt`) + `norm_stats.json` as a Kaggle Dataset

2. Edit the **Config** cell below to set `CHECKPOINT_PATH` and `NORM_STATS_PATH`3. Run All — generates `submission.csv` + prediction GeoTIFFs + visualizations

In [ ]:

# ── Install dependencies ──
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightning', 'einops', 'safetensors', 'huggingface_hub', 'rasterio',
    'scipy',
])

import timm, torch, lightning
print(f'torch     {torch.__version__}')
print(f'timm      {timm.__version__}')
print(f'lightning {lightning.__version__}')
print('✓ Ready — no terratorch needed.')


In [ ]:

import warnings; warnings.filterwarnings('ignore')

import os, json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import rasterio
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import timm
from scipy.ndimage import convolve as _sci_convolve

import lightning.pytorch as pl
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassAccuracy

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


---
## Configuration

Edit the cell below to point to your trained checkpoint and norm stats.

In [ ]:

# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit checkpoint and norm stats paths
# ═══════════════════════════════════════════════════════════════════

# ── Checkpoint & norm stats ──
CHECKPOINT_PATH = '/kaggle/input/models/anonymus11/300m-testing/pytorch/default/1/experiments/v2_300m_tl_6ch_sardb_nolee/checkpoints/best-epochepoch=35.ckpt'
NORM_STATS_PATH = '/kaggle/input/models/anonymus11/300m-testing/pytorch/default/1/experiments/v2_300m_tl_6ch_sardb_nolee/norm_stats.json'

# Local paths:
# CHECKPOINT_PATH = f'C:/Users/sorat/Downloads/aisehack-theme-1/outputs/v2_300m_tl_6ch_{_sar_tag}/checkpoints/best.ckpt'
# NORM_STATS_PATH = f'C:/Users/sorat/Downloads/aisehack-theme-1/outputs/v2_300m_tl_6ch_{_sar_tag}/norm_stats.json'

# ── Architecture constants (must match training — DO NOT CHANGE) ──
EMBED_DIM   = 1024              # ViT-Large (300M-TL)
DEPTH       = 24
OUT_INDICES = (5, 11, 17, 23)
PATCH_SIZE  = 16
TIMM_MODEL  = 'vit_large_patch16_224'
DECODER_CH  = 256
NUM_CLASSES = 2
HEAD_DROPOUT = 0.1
IMG_SIZE    = 224
CLASS_WEIGHTS = [0.25, 0.75]  # must match training checkpoint

# ── Inference settings ──
USE_SLIDING_WINDOW = True       # Recommended (IMG_SIZE < full 512 image)
SLIDE_STRIDE       = 112        # stride = IMG_SIZE // 2
USE_TTA            = True      # Test-time augmentation (h/v flips)

# ── Channels: raw 6 bands, SAR in dB (same as training) ──
RAW_BANDS = ['HH', 'HV', 'Green', 'Red', 'NIR', 'SWIR']

# ── SAR preprocessing toggle (MUST match training notebook!) ──
# True  = Refined Lee speckle filter → dB  (experiment: v2_300m_tl_6ch_sardb)
# False = dB only                          (experiment: v2_300m_tl_6ch_sardb_nolee)
USE_REFINED_LEE = False

_sar_tag = 'sardb' if USE_REFINED_LEE else 'sardb_nolee'
EXP_CFG = {
    'name': f'v2_300m_tl_6ch_{_sar_tag}',
    'desc': f'Prithvi-EO-2.0-300M-TL-Sen1Floods11 + UperNet, 6 bands (SAR {"Refined Lee + dB" if USE_REFINED_LEE else "dB only"})',
    'raw_bands': RAW_BANDS,
    'indices': [],
    'use_dem': False,
}

ch_names = EXP_CFG['raw_bands']
N_CH = len(ch_names)

# ── Dataset paths ──
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    DATA_ROOT = Path('/kaggle/input/competitions/aisehack-theme-1/data')
else:
    DATA_ROOT = Path('C:/Users/sorat/Downloads/aisehack-theme-1/data')

PRED_IMG_DIR = DATA_ROOT / 'prediction' / 'image'
SPLIT_DIR    = DATA_ROOT / 'split'
print(f'SAR preprocessing: {"Refined Lee + dB" if USE_REFINED_LEE else "dB only (no filter)"}')
with open(SPLIT_DIR / 'pred.txt') as f:
    pred_files = [l.strip() for l in f if l.strip()]

print(f'Model    : Prithvi-EO-2.0-300M-TL-Sen1Floods11')
print(f'Backbone : {TIMM_MODEL} (embed_dim={EMBED_DIM}, depth={DEPTH}, patch={PATCH_SIZE})')

print(f'Channels ({N_CH}): {ch_names}')

print(f'SAR dB transform: HH, HV')
print(f'TTA         : {"Yes" if USE_TTA else "No"}')

print(f'Checkpoint  : {CHECKPOINT_PATH}')
print(f'Sliding win : {"Yes" if USE_SLIDING_WINDOW else "No"} (window={IMG_SIZE}, stride={SLIDE_STRIDE})')
print(f'Pred patches: {len(pred_files)}')

---
## Channel Engineering

Exact copy from `stage2-prithvi-v2-300m-tl-flood.ipynb`. Must match for correct normalization.

In [ ]:

# ═══════════════════════════════════════════════════════════════════
#  Channel Engineering (same as training notebook)
# ═══════════════════════════════════════════════════════════════════

RAW_BAND_IDX = {'HH': 0, 'HV': 1, 'Green': 2, 'Red': 3, 'NIR': 4, 'SWIR': 5}
SAR_BANDS = {'HH', 'HV'}  # bands that get speckle filter + dB transform


# ── Refined Lee Speckle Filter ──────────────────────────────

def _build_direction_kernels(size=7):
    """Build 8 directional line kernels for the Refined Lee filter."""
    half = size // 2
    kernels = []
    for angle_deg in [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5]:
        k = np.zeros((size, size), dtype=np.float64)
        angle_rad = np.deg2rad(angle_deg)
        nx, ny = np.cos(angle_rad), np.sin(angle_rad)
        for i in range(size):
            for j in range(size):
                di, dj = i - half, j - half
                dist = abs(di * nx + dj * ny)
                if dist <= 1.0:
                    k[i, j] = 1.0
        k[half, half] = 1.0
        kernels.append(k)
    return kernels

_LEE_KERNELS_7 = _build_direction_kernels(7)


def refined_lee_filter(band, window_size=7, n_looks=1):
    """Refined Lee speckle filter for a single SAR intensity band.
    Operates in linear power domain (NOT dB)."""
    if window_size == 7:
        kernels = _LEE_KERNELS_7
    else:
        kernels = _build_direction_kernels(window_size)

    b = band.astype(np.float64)
    b_sq = b ** 2

    best_cv  = np.full(b.shape, np.inf, dtype=np.float64)
    best_mean = np.zeros_like(b)
    best_var  = np.zeros_like(b)

    for k in kernels:
        k_norm = k / k.sum()
        lm  = _sci_convolve(b,    k_norm, mode='reflect')
        lsq = _sci_convolve(b_sq, k_norm, mode='reflect')
        lv  = np.maximum(lsq - lm ** 2, 0.0)
        cv  = np.sqrt(lv) / (np.abs(lm) + 1e-10)

        better = cv < best_cv
        best_cv[better]   = cv[better]
        best_mean[better] = lm[better]
        best_var[better]  = lv[better]

    noise_var = (best_mean ** 2) / max(n_looks, 1.0)
    weight = np.clip((best_var - noise_var) / (best_var + 1e-10), 0.0, 1.0)
    result = best_mean + weight * (b - best_mean)
    return result.astype(np.float32)


# ── dB transform ────────────────────────────────────────────

def sar_to_db(x):
    """Convert SAR backscatter (linear power) to decibels."""
    return (10.0 * np.log10(np.clip(x, 1e-10, None))).astype(np.float32)


# ── Channel stack builder ──────────────────────────────────

def build_channel_stack(img6, dem, exp_cfg):
    '''Assemble the channel tensor for a given experiment config.
    SAR bands: optionally Refined Lee filter → dB scale, then stack.'''
    channels = []
    for b in exp_cfg['raw_bands']:
        band = img6[RAW_BAND_IDX[b]]
        if b in SAR_BANDS:
            if USE_REFINED_LEE:
                band = refined_lee_filter(band)  # speckle suppression (linear domain)
            band = sar_to_db(band)               # convert to dB
        channels.append(band)
    return np.stack(channels, axis=0).astype(np.float32)


print(f'Channel engineering defined ({"Refined Lee + dB" if USE_REFINED_LEE else "dB only"} on HH, HV).')
print(f'Bands: {list(RAW_BAND_IDX.keys())}')


---
## Model Architecture

Exact copy from `stage2-prithvi-v2-300m-tl-flood.ipynb`. Must match byte-for-byte for `load_state_dict` to succeed.

- ViT-Large encoder (1024-dim, 24 layers, patch_size=16)
- UperNet decoder (scale_modules=True, PSP + FPN)
- SegmentationHead (dropout + Conv2d)
- CombinedLoss (CE=0.2 + Dice=0.5 + Boundary=0.3)

In [ ]:

# ═══════════════════════════════════════════════════════════════════
#  Model Architecture — ViT-Large + UperNet + SegHead + AuxFCNHead
#
#  EXACT copy from stage2-prithvi-v2-300m-tl-flood.ipynb.
#  Must match training for checkpoint loading.
# ═══════════════════════════════════════════════════════════════════

from timm.models.vision_transformer import Block


# ── Smart Patch Embed Transfer (kept for state_dict compat) ──
BAND_TO_PRETRAINED_INDEX = {
    'Green': 1,
    'Red':   2,
    'NIR':   3,
    'SWIR':  4,
}


class ViTEncoder(nn.Module):
    """Wraps timm ViT-Large to extract multi-layer 2D feature maps.

    Extracts features at layers [7, 15, 23, 31] via forward hooks.
    """

    def __init__(self, timm_name, in_channels, img_size=224,
                 pretrained=False, out_indices=(5, 11, 17, 23)):
        super().__init__()
        self.vit = timm.create_model(
            timm_name, pretrained=pretrained,
            img_size=img_size, in_chans=in_channels, num_classes=0,
            dynamic_img_size=True,
        )
        self.embed_dim   = self.vit.embed_dim
        self.out_indices = out_indices
        pe = self.vit.patch_embed
        self.patch_size = (pe.patch_size[0]
                           if isinstance(pe.patch_size, (tuple, list))
                           else pe.patch_size)

        self._cache = {}
        for idx in out_indices:
            self.vit.blocks[idx].register_forward_hook(self._hook(idx))

    def _hook(self, idx):
        def fn(module, inp, out):
            self._cache[idx] = out
        return fn

    def forward(self, x):
        B, C, H, W = x.shape
        h, w = H // self.patch_size, W // self.patch_size

        self.vit.forward_features(x)

        feats = []
        for idx in self.out_indices:
            tok = self._cache[idx]
            if tok.shape[1] != h * w:
                tok = tok[:, -h * w:]
            feats.append(tok.transpose(1, 2).reshape(B, self.embed_dim, h, w))
        return feats


class ConvModule(nn.Module):
    """Conv2d + BatchNorm2d + ReLU."""
    def __init__(self, in_channels, out_channels, kernel_size=3,
                 padding=0, dilation=1, stride=1, inplace=False):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              padding=padding, dilation=dilation,
                              stride=stride, bias=False)
        self.norm = nn.BatchNorm2d(out_channels)
        self.act  = nn.ReLU(inplace=inplace)

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class PPM(nn.ModuleList):
    """Pooling Pyramid Module."""
    def __init__(self, pool_scales, in_channels, channels, align_corners=True):
        super().__init__()
        self.align_corners = align_corners
        for pool_scale in pool_scales:
            self.append(
                nn.Sequential(
                    nn.AdaptiveAvgPool2d(pool_scale),
                    ConvModule(in_channels, channels, 1, inplace=True),
                )
            )

    def forward(self, x):
        ppm_outs = []
        for ppm in self:
            ppm_out = ppm(x)
            upsampled = F.interpolate(
                ppm_out, size=x.size()[2:],
                mode='bilinear', align_corners=self.align_corners)
            ppm_outs.append(upsampled)
        return ppm_outs


class UperNetDecoder(nn.Module):
    """UperNet decoder with scale_modules for ViT backbones."""

    def __init__(self, embed_dim, pool_scales=(1, 2, 3, 6),
                 channels=256, align_corners=True, scale_modules=False):
        super().__init__()
        self.align_corners = align_corners
        self.out_channels  = channels
        self.channels      = channels

        if isinstance(embed_dim, int):
            embed_dim = [embed_dim] * 4

        self._use_scale_modules = scale_modules
        if scale_modules:
            self.fpn1 = nn.Sequential(
                nn.ConvTranspose2d(embed_dim[0], embed_dim[0] // 2, 2, 2),
                nn.BatchNorm2d(embed_dim[0] // 2),
                nn.GELU(),
                nn.ConvTranspose2d(embed_dim[0] // 2, embed_dim[0] // 4, 2, 2))
            self.fpn2 = nn.Sequential(
                nn.ConvTranspose2d(embed_dim[1], embed_dim[1] // 2, 2, 2))
            self.fpn3 = nn.Sequential(nn.Identity())
            self.fpn4 = nn.Sequential(nn.MaxPool2d(kernel_size=2, stride=2))
            self.embed_dim = [embed_dim[0] // 4, embed_dim[1] // 2,
                              embed_dim[2], embed_dim[3]]
        else:
            self.embed_dim = embed_dim

        self.psp_modules = PPM(
            pool_scales, self.embed_dim[-1], self.channels,
            align_corners=self.align_corners)
        self.bottleneck = ConvModule(
            self.embed_dim[-1] + len(pool_scales) * self.channels,
            self.channels, 3, padding=1, inplace=True)

        self.lateral_convs = nn.ModuleList()
        self.fpn_convs     = nn.ModuleList()
        for ed in self.embed_dim[:-1]:
            self.lateral_convs.append(
                ConvModule(ed, self.channels, 1, inplace=False))
            self.fpn_convs.append(
                ConvModule(self.channels, self.channels, 3, padding=1, inplace=False))

        self.fpn_bottleneck = ConvModule(
            len(self.embed_dim) * self.channels, self.channels, 3,
            padding=1, inplace=True)

    def psp_forward(self, inputs):
        x = inputs[-1]
        psp_outs = [x]
        psp_outs.extend(self.psp_modules(x))
        psp_outs = torch.cat(psp_outs, dim=1)
        return self.bottleneck(psp_outs)

    def forward(self, inputs, hw=None):
        if self._use_scale_modules:
            scaled = []
            scaled.append(self.fpn1(inputs[0]))
            scaled.append(self.fpn2(inputs[1]))
            scaled.append(self.fpn3(inputs[2]))
            scaled.append(self.fpn4(inputs[3]))
            inputs = scaled

        laterals = [lat_conv(inputs[i])
                    for i, lat_conv in enumerate(self.lateral_convs)]
        laterals.append(self.psp_forward(inputs))

        used_backbone_levels = len(laterals)
        for i in range(used_backbone_levels - 1, 0, -1):
            prev_shape = laterals[i - 1].shape[2:]
            laterals[i - 1] = laterals[i - 1] + F.interpolate(
                laterals[i], size=prev_shape,
                mode='bilinear', align_corners=self.align_corners)

        fpn_outs = [self.fpn_convs[i](laterals[i])
                    for i in range(used_backbone_levels - 1)]
        fpn_outs.append(laterals[-1])

        for i in range(used_backbone_levels - 1, 0, -1):
            fpn_outs[i] = F.interpolate(
                fpn_outs[i], size=fpn_outs[0].shape[2:],
                mode='bilinear', align_corners=self.align_corners)

        feats = self.fpn_bottleneck(torch.cat(fpn_outs, dim=1))
        return feats


class SegmentationHead(nn.Module):
    """Classification head: Identity → Dropout → Conv2d."""
    def __init__(self, in_channels, num_classes, dropout=0.0):
        super().__init__()
        dropout_layer = nn.Identity() if dropout == 0 else nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Identity(),
            dropout_layer,
            nn.Conv2d(in_channels, num_classes, 1),
        )

    def forward(self, x):
        return self.head(x)


class AuxFCNHead(nn.Module):
    """Lightweight auxiliary head for deep supervision (MMSeg-style).

    Conv2d(in, mid, 3) → BN → ReLU → Dropout → Conv2d(mid, n_cls, 1)
    """
    def __init__(self, in_channels, num_classes, mid_channels=256, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.cls  = nn.Conv2d(mid_channels, num_classes, 1)

    def forward(self, x):
        return self.cls(self.drop(self.conv(x)))


# ── Loss Functions (must match training for state_dict compat) ──

class DiceLoss(nn.Module):
    def __init__(self, num_classes=2, ignore_index=-1, smooth=1.0):
        super().__init__()
        self.num_classes  = num_classes
        self.ignore_index = ignore_index
        self.smooth       = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        valid = (targets != self.ignore_index)
        targets_clean = targets.clone()
        targets_clean[~valid] = 0
        one_hot = F.one_hot(targets_clean, self.num_classes).permute(0, 3, 1, 2).float()
        valid_mask = valid.unsqueeze(1).float()
        one_hot = one_hot * valid_mask
        probs   = probs   * valid_mask
        dims = (0, 2, 3)
        intersection = (probs * one_hot).sum(dims)
        union = probs.sum(dims) + one_hot.sum(dims)
        dice = 1.0 - (2.0 * intersection + self.smooth) / (union + self.smooth)
        return dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.register_buffer('weight', weight)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             ignore_index=self.ignore_index, reduction='none')
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


class BoundaryLoss(nn.Module):
    def __init__(self, num_classes=2, ignore_index=-1, kernel_size=5):
        super().__init__()
        self.num_classes  = num_classes
        self.ignore_index = ignore_index
        self.ks  = kernel_size
        self.pad = kernel_size // 2

    def _get_boundary_mask(self, targets):
        t = targets.clone().float()
        t[targets == self.ignore_index] = 0
        t = t.unsqueeze(1)
        pooled = F.avg_pool2d(t, self.ks, stride=1, padding=self.pad)
        boundary = (pooled.squeeze(1) != t.squeeze(1)).float()
        valid = (targets != self.ignore_index).float()
        return boundary * valid

    def forward(self, logits, targets):
        boundary = self._get_boundary_mask(targets)
        if boundary.sum() < 1:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        ce = F.cross_entropy(logits, targets, ignore_index=self.ignore_index,
                             reduction='none')
        return (ce * boundary).sum() / boundary.sum().clamp(min=1)


class CombinedLoss(nn.Module):
    def __init__(self, num_classes=2, class_weights=None, ignore_index=-1,
                 ce_weight=0.2, dice_weight=0.5, boundary_weight=0.3):
        super().__init__()
        w = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None
        self.ce_weight       = ce_weight
        self.dice_weight     = dice_weight
        self.boundary_weight = boundary_weight
        self.ce       = nn.CrossEntropyLoss(weight=w, ignore_index=ignore_index)
        self.dice     = DiceLoss(num_classes=num_classes, ignore_index=ignore_index)
        self.boundary = BoundaryLoss(num_classes=num_classes, ignore_index=ignore_index)

    def forward(self, logits, targets):
        return (self.ce_weight       * self.ce(logits, targets)
              + self.dice_weight     * self.dice(logits, targets)
              + self.boundary_weight * self.boundary(logits, targets))


# ═══════════════════════════════════════════════════════════════════
#  FloodSegModel — inference stub (architecture identical to training)
# ═══════════════════════════════════════════════════════════════════

class FloodSegModel(pl.LightningModule):
    """ViT-Large encoder + UperNet decoder + SegmentationHead + AuxFCNHead.

    Stripped down for inference — no backbone weight downloading.
    Architecture is byte-for-byte identical to the training notebook.
    """

    def __init__(self, in_channels, num_classes=2,
                 decoder_channels=256, head_dropout=0.1,
                 lr=3e-5, weight_decay=0.05, class_weights=None,
                 freeze_backbone=False, img_size=224,
                 warmup_iters=50, warmup_ratio=1e-6,
                 channel_names=None, transfer_mode='full'):
        super().__init__()
        self.save_hyperparameters(ignore=['channel_names'])

        # ── Encoder: ViT-Large ──
        self.encoder = ViTEncoder(
            timm_name=TIMM_MODEL,
            in_channels=in_channels,
            img_size=img_size,
            pretrained=False,
            out_indices=OUT_INDICES,
        )

        # ── Decoder: UperNet ──
        self.decoder = UperNetDecoder(
            embed_dim=EMBED_DIM,
            pool_scales=(1, 2, 3, 6),
            channels=decoder_channels,
            align_corners=True,
            scale_modules=True,
        )

        # ── Head ──
        self.seg_head = SegmentationHead(
            in_channels=decoder_channels,
            num_classes=num_classes,
            dropout=head_dropout,
        )

        # ── Auxiliary Head: on encoder feats[2] (block 17) for deep supervision ──
        self.aux_head = AuxFCNHead(
            in_channels=EMBED_DIM,
            num_classes=num_classes,
            mid_channels=256,
            dropout=head_dropout,
        )

        # ── Loss & Metrics (must match training for state_dict compat) ──
        self.loss_fn    = CombinedLoss(
            num_classes=num_classes, class_weights=class_weights,
            ignore_index=-1, ce_weight=0.2, dice_weight=0.5,
            boundary_weight=0.3)  # sum=1.0
        w = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None
        self.aux_loss_fn = nn.CrossEntropyLoss(weight=w, ignore_index=-1)
        self.train_miou = MulticlassJaccardIndex(num_classes, ignore_index=-1)
        self.val_miou   = MulticlassJaccardIndex(num_classes, ignore_index=-1)
        self.test_miou  = MulticlassJaccardIndex(num_classes, ignore_index=-1)
        self.test_acc   = MulticlassAccuracy(num_classes, ignore_index=-1)

    def forward(self, x):
        feats   = self.encoder(x)
        decoded = self.decoder(feats)
        logits  = self.seg_head(decoded)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, x.shape[2:],
                                   mode='bilinear', align_corners=True)
        return logits

    # Stubs required for Lightning to load checkpoint cleanly
    def training_step(self, batch, batch_idx):   pass
    def validation_step(self, batch, batch_idx): pass
    def test_step(self, batch, batch_idx):       pass
    def configure_optimizers(self):              pass


print('Architecture defined:')
print(f'  ViTEncoder  : {TIMM_MODEL} ({EMBED_DIM}-dim, {DEPTH} layers, patch={PATCH_SIZE})')
print('  UperNetDecoder: scale_modules + PSP + FPN')
print('  SegmentationHead: dropout + Conv2d')
print(f'  AuxFCNHead: on feats[2] (block {OUT_INDICES[2]})')
print('  CombinedLoss: CE=0.2 + Dice=0.5 + Boundary=0.3')


---
## Load Norm Stats + Build Model + Load Checkpoint

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  1. Load normalization stats saved during training
# ═══════════════════════════════════════════════════════════════════

with open(NORM_STATS_PATH) as f:
    stats = json.load(f)
means = np.array(stats['means'], dtype=np.float32).reshape(-1, 1, 1)
stds  = np.array(stats['stds'],  dtype=np.float32).reshape(-1, 1, 1)

print(f'Normalization stats ({len(stats["means"])} channels):')
for i, name in enumerate(ch_names):
    print(f'  {name:>6s}  mean={stats["means"][i]:>10.4f}  std={stats["stds"][i]:>10.4f}')

# ═══════════════════════════════════════════════════════════════════
#  2. Build model (same constructor args as training)
# ═══════════════════════════════════════════════════════════════════

model = FloodSegModel(
    in_channels      = N_CH,
    num_classes      = NUM_CLASSES,
    decoder_channels = DECODER_CH,
    head_dropout     = HEAD_DROPOUT,
    lr               = 3e-5,           # unused — configure_optimizers is a stub
    weight_decay     = 0.05,           # unused
    class_weights    = CLASS_WEIGHTS,
    freeze_backbone  = False,
    img_size         = IMG_SIZE,
    channel_names    = ch_names,
    transfer_mode    = 'full',
)

# ═══════════════════════════════════════════════════════════════════
#  3. Load trained weights from checkpoint (no HuggingFace download)
# ═══════════════════════════════════════════════════════════════════

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
state_dict = ckpt.get('state_dict', ckpt)

# Strip any 'model.' prefix (sometimes added by Lightning)
if any(k.startswith('model.') for k in state_dict):
    state_dict = {k.replace('model.', '', 1): v for k, v in state_dict.items()}

info = model.load_state_dict(state_dict, strict=False)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_p = sum(p.numel() for p in model.parameters())
print(f'\n✓ FloodSegModel loaded on {device} ({total_p:,} params)')
if info.missing_keys:
    print(f'  Missing keys   : {len(info.missing_keys)}')
    for k in info.missing_keys[:5]:
        print(f'    - {k}')
if info.unexpected_keys:
    print(f'  Unexpected keys: {len(info.unexpected_keys)}')
    for k in info.unexpected_keys[:5]:
        print(f'    - {k}')

---
## Training & Validation Curves

Load the CSVLogger metrics file and plot loss + mIoU over epochs.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Training & Validation Curves from CSVLogger
# ═══════════════════════════════════════════════════════════════════

# ── Locate the CSVLogger metrics file ──
# CSVLogger saves to: {exp_dir}/logs/version_X/metrics.csv
# The checkpoint path tells us exp_dir:
#   .../experiments/v2_300m_tl_6ch_sardb/checkpoints/best.ckpt
#   → exp_dir = parent of 'checkpoints'

ckpt_path = Path(CHECKPOINT_PATH)
exp_dir   = ckpt_path.parent.parent          # up from checkpoints/best.ckpt
logs_dir  = exp_dir / 'logs'

# Find the latest version folder
if logs_dir.exists():
    versions = sorted(logs_dir.glob('version_*'), key=lambda p: int(p.name.split('_')[1]))
    csv_path = versions[-1] / 'metrics.csv' if versions else None
else:
    csv_path = None

if csv_path is None or not csv_path.exists():
    print(f'⚠ No metrics.csv found under {logs_dir}')
    print('  Training curves will be available after training completes.')
else:
    raw = pd.read_csv(csv_path)
    print(f'Loaded {len(raw)} log rows from {csv_path}')

    # ── CSVLogger writes one row per log call; pivot by epoch ──
    train_loss = raw.dropna(subset=['train/loss']).groupby('epoch')['train/loss'].mean()
    train_miou = raw.dropna(subset=['train/mIoU']).groupby('epoch')['train/mIoU'].mean()

    val_loss = raw.dropna(subset=['val/loss']).groupby('epoch')['val/loss'].mean() \
               if 'val/loss' in raw.columns else pd.Series(dtype=float)
    val_miou = raw.dropna(subset=['val/mIoU']).groupby('epoch')['val/mIoU'].mean() \
               if 'val/mIoU' in raw.columns else pd.Series(dtype=float)
    val_biou = raw.dropna(subset=['val/boundary_mIoU']).groupby('epoch')['val/boundary_mIoU'].mean() \
               if 'val/boundary_mIoU' in raw.columns else pd.Series(dtype=float)

    test_miou = raw.dropna(subset=['test/mIoU']).groupby('epoch')['test/mIoU'].mean() \
                if 'test/mIoU' in raw.columns else pd.Series(dtype=float)

    # ═════════════ Figure 1: Loss ═════════════
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    if not train_loss.empty:
        ax.plot(train_loss.index, train_loss.values, 'o-', label='Train Loss', color='#2980b9', markersize=3)
    if not val_loss.empty:
        ax.plot(val_loss.index, val_loss.values, 's-', label='Val Loss', color='#e74c3c', markersize=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training & Validation Loss', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ═════════════ Figure 2: mIoU ═════════════
    ax = axes[1]
    if not train_miou.empty:
        ax.plot(train_miou.index, train_miou.values, 'o-', label='Train mIoU', color='#2980b9', markersize=3)
    if not val_miou.empty:
        ax.plot(val_miou.index, val_miou.values, 's-', label='Val mIoU', color='#e74c3c', markersize=3)
    if not val_biou.empty:
        ax.plot(val_biou.index, val_biou.values, '^--', label='Val Boundary mIoU', color='#e67e22', markersize=3)
    if not test_miou.empty:
        ax.plot(test_miou.index, test_miou.values, 'D--', label='Test mIoU', color='#27ae60', markersize=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mIoU')
    ax.set_title('mIoU Over Epochs', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

    fig.suptitle(f'Training Curves — {EXP_CFG["name"]}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Print best metrics ──
    if not val_miou.empty:
        best_epoch = val_miou.idxmax()
        print(f'\n★ Best val/mIoU = {val_miou.max():.4f}  (epoch {best_epoch})')
        if not val_loss.empty and best_epoch in val_loss.index:
            print(f'  val/loss at best = {val_loss[best_epoch]:.4f}')
        if not val_biou.empty and best_epoch in val_biou.index:
            print(f'  val/boundary_mIoU at best = {val_biou[best_epoch]:.4f}')
    if not test_miou.empty:
        print(f'  Best test/mIoU = {test_miou.max():.4f}  (epoch {test_miou.idxmax()})')
    if not train_loss.empty:
        print(f'  Final train/loss = {train_loss.iloc[-1]:.4f}  (epoch {train_loss.index[-1]})')

    print(f'\nTotal epochs logged: {int(raw["epoch"].max()) + 1}')

---
## Inference Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Inference utilities
# ═══════════════════════════════════════════════════════════════════

def mask_to_rle(mask):
    """Convert binary 2D mask → RLE string (column-major, 1-indexed, Kaggle format)."""
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def sliding_window_inference(model_fn, image_tensor, window_size, stride,
                              device, num_classes=2):
    """Sliding-window inference — averages *softmax probabilities* across overlapping crops.

    Averaging probabilities (not raw logits) gives correct uncertainty weighting
    when crops overlap, avoiding logit-scale bias at window edges.
    """
    C, H, W = image_tensor.shape
    prob_sum = torch.zeros(num_classes, H, W, device='cpu')
    count    = torch.zeros(H, W, device='cpu')

    for y in range(0, H, stride):
        for x in range(0, W, stride):
            y_end   = min(y + window_size, H)
            x_end   = min(x + window_size, W)
            y_start = max(0, y_end - window_size)
            x_start = max(0, x_end - window_size)

            crop = image_tensor[:, y_start:y_end, x_start:x_end].unsqueeze(0).to(device)
            with torch.no_grad():
                out = model_fn(crop).squeeze(0).cpu()
            # probs = out.softmax(0)          # ← convert to probs before accumulating
            prob_sum[:, y_start:y_end, x_start:x_end] += out
            count[y_start:y_end, x_start:x_end]       += 1

    prob_sum /= count.unsqueeze(0).clamp(min=1)
    return prob_sum.argmax(0).numpy()


def tta_inference(model_fn, image_tensor, window_size, stride, device,
                  num_classes=2, use_sliding=True):
    """Test-time augmentation: average softmax probabilities over 4 flip variants."""
    C, H, W = image_tensor.shape
    all_probs = torch.zeros(num_classes, H, W, device='cpu')

    augments = [
        lambda x: x,
        lambda x: x.flip(-1),
        lambda x: x.flip(-2),
        lambda x: x.flip(-1).flip(-2),
    ]
    de_augments = [
        lambda x: x,
        lambda x: x.flip(-1),
        lambda x: x.flip(-2),
        lambda x: x.flip(-1).flip(-2),
    ]

    for aug_fn, deaug_fn in zip(augments, de_augments):
        aug_img  = aug_fn(image_tensor)
        prob_sum = torch.zeros(num_classes, H, W, device='cpu')
        count    = torch.zeros(H, W, device='cpu')

        if use_sliding:
            for y in range(0, H, stride):
                for x in range(0, W, stride):
                    y_end   = min(y + window_size, H)
                    x_end   = min(x + window_size, W)
                    y_start = max(0, y_end - window_size)
                    x_start = max(0, x_end - window_size)
                    crop = aug_img[:, y_start:y_end, x_start:x_end].unsqueeze(0).to(device)
                    with torch.no_grad():
                        out = model_fn(crop).squeeze(0).cpu()
                    # probs = out.softmax(0)
                    prob_sum[:, y_start:y_end, x_start:x_end] += out
                    count[y_start:y_end, x_start:x_end]       += 1
            prob_sum /= count.unsqueeze(0).clamp(min=1)
        else:
            with torch.no_grad():
                out = model_fn(aug_img.unsqueeze(0).to(device)).squeeze(0).cpu()
            prob_sum = out

        all_probs += deaug_fn(prob_sum)

    all_probs /= len(augments)
    return all_probs.argmax(0).numpy()


print('Inference functions defined:')
print('  mask_to_rle()              — binary mask → Kaggle RLE')
print('  sliding_window_inference() — softmax probability averaging across crops')
print('  tta_inference()            — flip TTA with softmax averaging')

---
## Run Inference & Generate Submission

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Run prediction on all patches and create submission.csv
# ═══════════════════════════════════════════════════════════════════

rows      = []
all_preds = {}   # fname → binary pred array (for viz + GeoTIFF saving)

print(f'Running inference on {len(pred_files)} patches...')
print(f'  Mode: {"sliding window" if USE_SLIDING_WINDOW else "full-image"}'
      f'{" + TTA" if USE_TTA else ""}')
print()

t0 = time.time()

with torch.no_grad():
    for idx, fname in enumerate(pred_files):
        # ── Load 6-band image ──
        with rasterio.open(PRED_IMG_DIR / f'{fname}_image.tif') as src:
            img6 = src.read().astype(np.float32)

        # ── Build channel stack + normalize ──
        ch = build_channel_stack(img6, None, EXP_CFG)
        ch = (ch - means) / (stds + 1e-8)
        ch_tensor = torch.from_numpy(ch).float()

        # ── Inference ──
        if USE_TTA:
            pred = tta_inference(
                model, ch_tensor, IMG_SIZE, SLIDE_STRIDE, device,
                num_classes=NUM_CLASSES, use_sliding=USE_SLIDING_WINDOW)
        elif USE_SLIDING_WINDOW:
            pred = sliding_window_inference(
                model, ch_tensor, IMG_SIZE, SLIDE_STRIDE, device,
                num_classes=NUM_CLASSES)
        else:
            tensor = ch_tensor.unsqueeze(0).to(device)
            logits = model(tensor)
            pred = logits.argmax(dim=1).squeeze(0).cpu().numpy()

        binary = (pred > 0).astype(np.uint8)
        all_preds[fname] = binary

        rle = mask_to_rle(binary)
        sample_id = fname.replace('_image', '')
        rows.append({'id': sample_id, 'rle_mask': rle})

        flood_pct = binary.sum() / binary.size * 100
        print(f'  [{idx+1:3d}/{len(pred_files)}] {sample_id}: {flood_pct:5.1f}% flood')

elapsed = time.time() - t0
print(f'\nInference complete in {elapsed:.1f}s ({elapsed/len(pred_files):.2f}s/patch)')

# ── Save submission.csv ──
df = pd.DataFrame(rows)
df['rle_mask'] = df['rle_mask'].replace('', ' ')

output_csv = '/kaggle/working/submission.csv' if ON_KAGGLE else 'submission.csv'
df.to_csv(output_csv, index=False)
print(f'✓ Saved {output_csv} ({len(rows)} patches)')

---
## Save Prediction GeoTIFFs

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Save georeferenced prediction TIFs
# ═══════════════════════════════════════════════════════════════════

tif_dir = Path('/kaggle/working/pred_tifs') if ON_KAGGLE else Path('pred_tifs')
tif_dir.mkdir(exist_ok=True)

for fname, pred_mask in all_preds.items():
    ref_path = PRED_IMG_DIR / f'{fname}_image.tif'
    with rasterio.open(ref_path) as src:
        meta = src.meta.copy()
    meta.update({'count': 1, 'dtype': 'int16', 'nodata': -1, 'compress': 'lzw'})
    out_path = tif_dir / f'{fname}_pred.tif'
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(pred_mask.astype(np.int16), 1)

print(f'✓ {len(all_preds)} prediction GeoTIFFs saved to {tif_dir}')

---
## Visual Sanity Check

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Visualize: input (Green band) | SAR (HH) | flood prediction
# ═══════════════════════════════════════════════════════════════════

flood_cmap = mcolors.ListedColormap(['#2c3e50', '#e74c3c'])
n_show = min(8, len(pred_files))

fig, axes = plt.subplots(n_show, 3, figsize=(14, 3.5 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for i in range(n_show):
    fname = pred_files[i]
    with rasterio.open(PRED_IMG_DIR / f'{fname}_image.tif') as src:
        img6 = src.read().astype(np.float32)

    # Column 0: Green band (optical context)
    green = img6[RAW_BAND_IDX['Green']]
    axes[i, 0].imshow(green, cmap='YlGn')
    axes[i, 0].set_ylabel(fname, fontsize=8)
    if i == 0:
        axes[i, 0].set_title('Input (Green)', fontweight='bold')
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])

    # Column 1: HH SAR
    hh = img6[RAW_BAND_IDX['HH']]
    axes[i, 1].imshow(hh, cmap='gray')
    if i == 0:
        axes[i, 1].set_title('Input (HH SAR)', fontweight='bold')
    axes[i, 1].set_xticks([]); axes[i, 1].set_yticks([])

    # Column 2: Flood prediction
    pred = all_preds[fname]
    axes[i, 2].imshow(pred, cmap=flood_cmap, vmin=0, vmax=1)
    flood_pct = pred.sum() / pred.size * 100
    if i == 0:
        axes[i, 2].set_title('Flood Prediction', fontweight='bold')
    axes[i, 2].text(pred.shape[1] // 2, pred.shape[0] - 20,
                    f'{flood_pct:.1f}% flood', ha='center', fontsize=9,
                    color='white',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    axes[i, 2].set_xticks([]); axes[i, 2].set_yticks([])

fig.suptitle(f'Prithvi-EO-2.0-300M-TL-Sen1Floods11 | v2_300m_tl_6ch_{_sar_tag}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

out_vis = '/kaggle/working/predictions_visual.png' if ON_KAGGLE else 'predictions_visual.png'
plt.savefig(out_vis, dpi=120, bbox_inches='tight')
plt.show()
print(f'✓ Saved {out_vis}')

---
## Submission Statistics & Verification

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Final verification
# ═══════════════════════════════════════════════════════════════════

df_check = pd.read_csv(output_csv)

print(f'Submission file : {output_csv}')
print(f'Total rows      : {len(df_check)}')
print(f'Columns         : {list(df_check.columns)}')
print(f'Sample IDs      : {df_check["id"].tolist()[:5]}{"..." if len(df_check) > 5 else ""}')

flood_coverages = [all_preds[fname].sum() / all_preds[fname].size * 100
                   for fname in pred_files]

print(f'\nFlood coverage statistics:')
print(f'  Mean   : {np.mean(flood_coverages):.1f}%')
print(f'  Median : {np.median(flood_coverages):.1f}%')
print(f'  Min    : {np.min(flood_coverages):.1f}%')
print(f'  Max    : {np.max(flood_coverages):.1f}%')
print(f'  >0%    : {sum(c > 0 for c in flood_coverages)}/{len(flood_coverages)} patches have flood')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(flood_coverages, bins=20, color='#3498db', edgecolor='white', alpha=0.8)
ax.set_xlabel('Flood coverage (%)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Flood Coverage Across Prediction Patches', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n✓ Ready to submit! Upload {output_csv} to Kaggle.')